# Colab Inference Server

Exposes two HTTP endpoints via ngrok for the campus navigation app:
- `/analyze`: image analysis using PaliGemma 2 (3B mix 448px)
- `/transcribe`: speech-to-text using Groq Whisper large-v3

**Run order:** execute cells top to bottom. Cell 1 restarts the runtime automatically. This is expected, then continue from cell 2.

In [ ]:
# Install dependencies.
# Pillow is pinned to avoid a known Colab runtime conflict with the _Ink module.
# The runtime is killed at the end so the new Pillow version loads cleanly on restart.
!pip install -q "Pillow==10.4.0"
!pip install -q -U \
    "transformers>=4.47.0" \
    torchvision \
    accelerate \
    sentencepiece \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok \
    groq \
    huggingface_hub \
    nest_asyncio

import os
os.kill(os.getpid(), 9)

## 1. Credentials

In [ ]:
import os
from google.colab import userdata

# Load secrets from Colab's secret manager (key icon in the left sidebar).
# Required secrets: HF_TOKEN, GROQ_API_KEY, NGROK_TOKEN
hfToken    = userdata.get("HF_TOKEN")
groqApiKey = userdata.get("GROQ_API_KEY")
ngrokToken = userdata.get("NGROK_TOKEN")

if not hfToken:    raise ValueError("Missing HF_TOKEN. Add it in Colab Secrets")
if not groqApiKey: raise ValueError("Missing GROQ_API_KEY. Add it in Colab Secrets")
if not ngrokToken: raise ValueError("Missing NGROK_TOKEN. Add it in Colab Secrets")

os.environ["HF_TOKEN"]         = hfToken
os.environ["GROQ_API_KEY"]     = groqApiKey
os.environ["NGROK_AUTH_TOKEN"] = ngrokToken

print("Secrets loaded.")

## 2. Hugging Face Login

In [ ]:
from huggingface_hub import login, whoami

# Explicit login required to access gated models (PaliGemma 2 requires approval).
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

## 3. Inference Server

In [ ]:
%%writefile server.py
import os
import io
import tempfile
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from groq import Groq

# Credentials
groqClient = Groq(api_key=os.environ["GROQ_API_KEY"])
hfToken    = os.environ["HF_TOKEN"]

# Device setup: use GPU (bfloat16) when available, CPU (float32) otherwise
modelId = "google/paligemma2-3b-mix-448"
device  = "cuda" if torch.cuda.is_available() else "cpu"
dtype   = torch.bfloat16 if device == "cuda" else torch.float32

print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Load PaliGemma 2 (takes 3 to 5 minutes on first run)
print(f"\nLoading {modelId} ...")
processor = AutoProcessor.from_pretrained(modelId, token=hfToken)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    modelId,
    torch_dtype=dtype,
    device_map="auto",
    token=hfToken,
    revision="main",
).eval()
print("PaliGemma 2 loaded.")

if device == "cuda":
    usedVram  = torch.cuda.memory_allocated() / 1024**3
    totalVram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM used: {usedVram:.1f} / {totalVram:.1f} GB")

# Domain prompt: biases Whisper towards campus-specific vocabulary.
# Listing known room codes, building names and POIs improves recognition
# of domain terms that a general STT model might otherwise mishear.
WHISPER_DOMAIN_PROMPT = (
    "UFV campus navigation. "
    "Buildings: Building G, Building CD, Building H. "
    "Floors: ground floor, first floor, second floor, basement, Agora. "
    "Classrooms: 0.1, 0.2, 0.3, 0.4, 0.5, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 2.1. "
    "Places: elevator, stairs, cafeteria, gym, restroom, bathroom, "
    "AI laboratory, AI lab, Telefonica classroom, computer lab, "
    "secretary office, mentoring room, Pastoral office, "
    "Agora classroom, main entrance, directory sign. "
    "Commands: yes, no, done, repeat, cancel, I am lost, I am in a hurry, "
    "I want to go to, take me to, navigate to, I am in, I am at."
)

app = FastAPI(title="Campus Navigation Inference Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
async def root():
    return {"status": "running", "model": modelId, "device": device}


@app.post("/analyze")
async def analyzeImage(
    file: UploadFile = File(...),
    prompt: str = Form(...),
    max_tokens: str = Form(None),
):
    """Run a prompt and image through PaliGemma 2. Returns the model response as plain text."""
    try:
        imageBytes = await file.read()
        image = Image.open(io.BytesIO(imageBytes)).convert("RGB")
        maxNewTokens = int(max_tokens) if (max_tokens and max_tokens.isdigit()) else 256

        # PaliGemma 2 requires the <image> token at the start of the prompt
        if not prompt.strip().startswith("<image>"):
            prompt = "<image>\n" + prompt

        print(f"\n[/analyze] prompt={prompt[:120]!r}")

        inputs = processor(
            text=prompt,
            images=image,
            padding="longest",
            return_tensors="pt",
        ).to(model.device)
        inputs = inputs.to(dtype=model.dtype)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=maxNewTokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=(
                    processor.tokenizer.pad_token_id
                    or processor.tokenizer.eos_token_id
                ),
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens (skip the input prompt)
        inputLen = inputs["input_ids"].shape[-1]
        response = processor.decode(
            outputs[0][inputLen:], skip_special_tokens=True
        ).strip()

        print(f"[/analyze] response={response!r}")
        return {"success": True, "response": response, "error": None}

    except Exception as e:
        print(f"[/analyze] ERROR: {e}")
        return {"success": False, "response": "", "error": str(e)}


@app.post("/transcribe")
async def transcribeAudio(file: UploadFile = File(...)):
    """Transcribe audio using Groq Whisper large-v3. Returns the transcribed text."""
    try:
        audioData = await file.read()
        fname = file.filename or "audio.wav"
        print(f"\n[/transcribe] file={fname} size={len(audioData)} bytes")

        # Write to a temp file: Groq SDK requires a file path, not raw bytes
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            tmp.write(audioData)
            tmpPath = tmp.name

        try:
            with open(tmpPath, "rb") as audioFile:
                result = groqClient.audio.transcriptions.create(
                    file=audioFile,
                    model="whisper-large-v3",
                    response_format="text",
                    language="en",
                    temperature=0.0,
                    prompt=WHISPER_DOMAIN_PROMPT,
                )
            text = result.strip() if isinstance(result, str) else ""
        finally:
            os.unlink(tmpPath)

        print(f"[/transcribe] text={text!r}")
        return {"success": True, "text": text, "error": None}

    except Exception as e:
        print(f"[/transcribe] ERROR: {e}")
        return {"success": False, "text": "", "error": str(e)}


## 4. ngrok Tunnel

In [ ]:
import time
from pyngrok import ngrok, conf

PORT = 8000

conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]

# Close any existing tunnels before opening a new one
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception:
    pass
ngrok.kill()
time.sleep(2)

tunnel = ngrok.connect(PORT)
publicUrl = tunnel.public_url

print("=" * 55)
print(f"  Public URL: {publicUrl}")
print(f"\n  Set this in your Windows .env file:")
print(f"  COLAB_API_URL={publicUrl}")
print("=" * 55)
print("\n  Run the next cell to start the server.")

## 5. Start Server

This cell runs indefinitely while the server is active.
To stop it: **Runtime > Interrupt execution**.

In [ ]:
import nest_asyncio
import uvicorn

# nest_asyncio allows uvicorn to run inside Colab's existing event loop
nest_asyncio.apply()

print("Starting server...")
print("PaliGemma 2 takes 3 to 5 minutes to load on first run.")
print("Wait for 'Application startup complete.' before using the app.\n")

config = uvicorn.Config("server:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

In [ ]:
# Run this cell to verify the server is reachable.
# Only execute after seeing "Application startup complete." in the cell above.
import requests

try:
    response = requests.get(f"{publicUrl}/", timeout=5)
    print("Server ready:", response.json())
except Exception as e:
    print(f"Not ready yet, wait a moment and try again: {e}")